# Object Detection with KerasCV — Student Worksheet

**Total: 100 points**

You will implement:
- Loading and visualising images from a provided COCO subset
- Running a pre-trained YOLOv8 Pascal VOC detector and controlling NMS
- Implementing Intersection over Union (IoU) from scratch
- Parsing ground-truth annotations and building a tf.data evaluation pipeline
- Computing COCO evaluation metrics with `BoxCOCOMetrics`
- Computing AP@0.50 for the `person` class step by step

## Important rules

Use the **exact variable names** requested in each part — the grader depends on them.

Keep all **sanity-check `print(...)` lines** in place; do not remove them.

When a cell starts with:
```python
variable = None  # <-- replace
```
replace `None` with your solution.

Do **not** use `raise NotImplementedError`.

In [ ]:
import json
import numpy as np
import pandas as pd
import tensorflow as tf
import keras_cv
import matplotlib.pyplot as plt
from collections import defaultdict

np.random.seed(42)
tf.random.set_seed(42)

print('TensorFlow :', tf.__version__)
print('KerasCV    :', keras_cv.__version__)

In [ ]:
# ── Fill in your path ────────────────────────────────────────────────────────
IMG_DIR    = 'coco/'               # folder containing the 10 COCO .jpg images
ANNOT_FILE = 'coco/annotations.json'   # provided alongside the worksheet
# ─────────────────────────────────────────────────────────────────────────────

IMG_SIZE = (640, 640)
BBOX_FMT = 'xyxy'
N_IMGS   = 10

# load annotation file once — keys are the image filenames
ann    = json.load(open(ANNOT_FILE))
fnames = sorted(ann.keys())
print(f'{len(fnames)} images:', fnames)

In [ ]:
CLASS_MAP = {
     0: 'aeroplane',  1: 'bicycle',    2: 'bird',       3: 'boat',
     4: 'bottle',     5: 'bus',        6: 'car',        7: 'cat',
     8: 'chair',      9: 'cow',       10: 'diningtable',11: 'dog',
    12: 'horse',     13: 'motorbike', 14: 'person',     15: 'pottedplant',
    16: 'sheep',     17: 'sofa',      18: 'train',      19: 'tvmonitor',
}
CMAP_INV = {v: k for k, v in CLASS_MAP.items()}   # name → id


def set_nms(mdl, iou_thresh=0.5, conf_thresh=0.2, max_det=50):
    """Replace the NMS decoder of mdl with the given thresholds."""
    mdl.prediction_decoder = keras_cv.layers.NonMaxSuppression(
        bounding_box_format=BBOX_FMT,
        from_logits=False,
        max_detections=max_det,
        iou_threshold=iou_thresh,
        confidence_threshold=conf_thresh,
    )
    print(f'NMS: iou_threshold={iou_thresh}  confidence_threshold={conf_thresh}')

## Part 1 — Pre-trained YOLOv8 Inference (35 pts)

### 1.1 Load Images  *(10 pts)*

The filenames are already in `fnames` (sorted keys of `ann`).
Load each image from `IMG_DIR`, decode it, cast to `float32`,
resize with padding to `(640, 640)`, and stack into a single tensor.

Required variables:
- `images` — `tf.Tensor` of shape `(10, 640, 640, 3)`, dtype `float32`, range `[0, 255]`

In [ ]:
# TODO (1.1): load images
# Use fnames and IMG_DIR — no need to scan the directory
# Required: images

images = None  # <-- replace

# Sanity checks (do not remove)
print('images shape:', None if images is None else images.shape)
print('images dtype:', None if images is None else images.dtype)

### 1.2 Visualise Images

Display all 10 images in a 5 × 2 grid. Label each with its filename.

In [ ]:
# TODO (1.2): visualise all 10 images in a 5×2 grid

# YOUR CODE HERE

### 1.3 Load Model and Run Predictions  *(15 pts)*

Load the **YOLOv8-M Pascal VOC** pre-trained model and run inference on `images`.

Required variables:
- `model`      — `YOLOV8Detector` from preset `'yolo_v8_m_pascalvoc'`
- `detections` — dict with keys `'boxes'`, `'confidence'`, `'classes'`

In [ ]:
# TODO (1.3): load model and predict
# Required: model, detections

model      = None  # <-- replace
detections = None  # <-- replace

# Sanity checks (do not remove)
print('model type       :', type(model).__name__)
print('boxes shape      :', None if detections is None else detections['boxes'].shape)
print('confidence shape :', None if detections is None else detections['confidence'].shape)

### 1.4 Visualise Detections

Draw predicted bounding boxes on all 10 images (5 × 2 grid).
Show only detections with confidence > 0.2. Label each box with class name and score.

In [ ]:
# TODO (1.4): visualise detections (confidence > 0.2)

# YOUR CODE HERE

### 1.5 Fewer Bounding Boxes  *(5 pts)*

Use `set_nms` to reconfigure the model so that **fewer** bounding boxes survive,
then re-run `model.predict`.

Hint: raising `conf_thresh` discards low-confidence boxes;
lowering `iou_thresh` makes NMS more aggressive.

Required variable: `det_few`

In [ ]:
# TODO (1.5): stricter NMS → det_few

det_few = None  # <-- replace

print('det_few boxes shape:', None if det_few is None else det_few['boxes'].shape)

### 1.6 More Bounding Boxes  *(5 pts)*

Use `set_nms` to let **more** bounding boxes survive, then re-run.

Required variable: `det_many`

In [ ]:
# TODO (1.6): permissive NMS → det_many

det_many = None  # <-- replace

print('det_many boxes shape:', None if det_many is None else det_many['boxes'].shape)

### 1.7 Visualise the Comparison  *(optional)*

Show `det_few` and `det_many` side by side for one image of your choice.

In [ ]:
# (optional)

# YOUR CODE HERE

## Part 2 — Intersection over Union  *(10 pts)*

IoU (Intersection over Union) is the ratio of the overlapping area to the
combined area of two bounding boxes.  It is used in two places in this worksheet:

* **NMS** suppresses a box when its IoU with a higher-confidence box exceeds
  `iou_threshold`.
* **TP / FP assignment** (Part 4.4): a prediction counts as a True Positive only
  when its IoU with an unmatched GT box is ≥ `IOU_THR = 0.50`.

The helper `box_iou_matrix` (given below) computes pairwise IoU between two
sets of boxes.  Apply it to the two example predictions below and decide
whether each one would be a **TP** or **FP** at the standard threshold.

In [ ]:
# Given — do not modify ──────────────────────────────────────────────────────
def box_iou_matrix(a, b):
    """Pairwise IoU between arrays a (M,4) and b (N,4) in xyxy format."""
    x1    = np.maximum(a[:, 0:1], b[:, 0]);  y1 = np.maximum(a[:, 1:2], b[:, 1])
    x2    = np.minimum(a[:, 2:3], b[:, 2]);  y2 = np.minimum(a[:, 3:4], b[:, 3])
    inter = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)


# Example boxes (640×640 letterboxed image space, xyxy)
GT_BOX = np.array([[ 50., 100., 250., 400.]])   # one GT person box
PRED_A = np.array([[ 60., 110., 240., 390.]])   # prediction A — large overlap
PRED_B = np.array([[300., 100., 500., 400.]])   # prediction B — no overlap

iou_A = float(box_iou_matrix(PRED_A, GT_BOX)[0, 0])
iou_B = float(box_iou_matrix(PRED_B, GT_BOX)[0, 0])
print(f'IoU(PRED_A, GT_BOX) = {iou_A:.4f}')
print(f'IoU(PRED_B, GT_BOX) = {iou_B:.4f}')
# ─────────────────────────────────────────────────────────────────────────────

# TODO (2.1): set is_tp_A and is_tp_B (True / False) for IOU_THR = 0.50
IOU_THR = 0.50

is_tp_A = None  # <-- replace
is_tp_B = None  # <-- replace

# Sanity checks (do not remove)
_fmt = lambda v: 'TP' if v else ('FP' if v is not None else '?')
print(f'PRED_A  iou={iou_A:.4f}  →  {_fmt(is_tp_A)}')
print(f'PRED_B  iou={iou_B:.4f}  →  {_fmt(is_tp_B)}')

## Part 3 — Ground-Truth Annotations  *(20 pts)*

### 3.1 Parse the Annotations  *(10 pts)*

`ann` is already loaded (see config cell). Each entry looks like:
```python
ann['000000031599.jpg'] = {
    'boxes':   [[x1, y1, x2, y2], ...],
    'classes': ['boat', 'boat', ...]
}
```

Required variables:
- `gt_boxes`   — list of 10 inner lists, each `[[x1,y1,x2,y2], …]` as **floats**
- `gt_classes` — list of 10 inner lists, each containing **integer** class ids
  (use `CMAP_INV` to convert name → id; keep the same order as `fnames`)

In [ ]:
# TODO (3.1): parse gt_boxes and gt_classes from ann
# Required: gt_boxes, gt_classes

gt_boxes   = None  # <-- replace
gt_classes = None  # <-- replace

# Sanity checks (do not remove)
print('gt_boxes[0] count:', None if gt_boxes   is None else len(gt_boxes[0]))
print('gt_classes[0]    :', None if gt_classes is None else gt_classes[0][:4])

### 3.2 Per-Class GT Count Table  *(10 pts)*

Count total GT boxes per class across all 10 images and display as a sorted table.

Required variable:
- `class_counts` — `dict` mapping class **name** → total GT box count (classes with count > 0)

In [ ]:
# TODO (3.2): build class_counts and display table
# Required: class_counts (dict)

class_counts = None  # <-- replace

if class_counts is not None:
    _df = pd.DataFrame(sorted(class_counts.items(), key=lambda x: -x[1]),
                       columns=['Class', 'GT boxes'])
    display(_df)
print('class_counts type:', type(class_counts))

## Part 4 — Evaluation Pipeline  *(35 pts)*

### 4.1 Build the tf.data Evaluation Pipeline  *(10 pts)*

Use the helpers below to build a pipeline that reads each image + GT from disk,
applies the KerasCV `Resizing` layer (which also transforms the bounding boxes),
and collects everything into a single batch of 10.

Required variables:
- `I_ev` — `float32` tensor of shape `(10, 640, 640, 3)`
- `b_ev` — dict with keys `'boxes'` and `'classes'` (both `RaggedTensor`)

In [ ]:
# Given helpers — do not modify
imResize = keras_cv.layers.Resizing(
    *IMG_SIZE, pad_to_aspect_ratio=True, bounding_box_format=BBOX_FMT)

def _load_image_bb(path, boxes, classes):
    raw = tf.io.read_file(path)
    img = tf.cast(tf.image.decode_jpeg(raw, channels=3), tf.float32)
    return img, {'boxes': boxes, 'classes': classes}

def _resize_image_bb(img, bb):
    d = imResize({'images': img, 'bounding_boxes': bb})
    return d['images'], d['bounding_boxes']

In [ ]:
# TODO (4.1): build the tf.data pipeline → I_ev, b_ev
# Use fnames, gt_boxes, gt_classes
# Use tf.ragged.constant for boxes and classes
# Pipeline: from_tensor_slices → map(_load_image_bb) → ragged_batch(N_IMGS)
#           → map(_resize_image_bb) → cache
# Extract one batch: I_ev, b_ev = next(iter(ds_eval))
# Required: I_ev, b_ev

I_ev = None  # <-- replace
b_ev = None  # <-- replace

print('I_ev shape      :', None if I_ev is None else I_ev.shape)
print('b_ev boxes type :', None if b_ev is None else type(b_ev['boxes']).__name__)

### 4.2 Visualise GT vs Predicted

Display all 10 images with **green** GT boxes and **orange** predictions (confidence > 0.2).

In [ ]:
# TODO (4.2): GT (green) vs predictions (orange) for all 10 images
set_nms(model, iou_thresh=0.5, conf_thresh=0.2)
preds_viz = model.predict(I_ev, verbose=0)

# YOUR CODE HERE

### 4.3 COCO Evaluation Metrics  *(15 pts)*

Use `keras_cv.metrics.BoxCOCOMetrics` to compute mAP on the 10-image set.
Because images have different numbers of GT boxes, pad them to a fixed dense
shape with the helper `pad_gt_to_fixed` provided below.

Required variables:
- `model_eval`   — fresh `YOLOV8Detector` (same preset)
- `coco_metric`  — `BoxCOCOMetrics` instance after calling `update_state`
- `eval_results` — dict from `coco_metric.result()`
- `mAP50`        — `float`, value at key `'MaP@.50IOU'`

In [ ]:
# Given — do not modify
def pad_gt_to_fixed(boxes_ragged, classes_ragged, max_boxes=100):
    """Pad ragged GT tensors to fixed dense shape required by BoxCOCOMetrics."""
    boxes_d   = boxes_ragged.to_tensor(default_value=-1.,
                                        shape=[None, max_boxes, 4])
    classes_d = classes_ragged.to_tensor(default_value=-1.,
                                          shape=[None, max_boxes])
    return boxes_d, classes_d

In [ ]:
# TODO (4.3): compute COCO metrics
# 1. Load a fresh YOLOV8Detector (same preset) → model_eval
# 2. model_eval.predict(I_ev) → preds_eval
# 3. pad_gt_to_fixed(b_ev['boxes'], b_ev['classes']) → boxes_d, classes_d
# 4. y_true = {'boxes': boxes_d, 'classes': classes_d}
# 5. BoxCOCOMetrics(bounding_box_format=BBOX_FMT, evaluate_freq=1)
# 6. coco_metric.update_state(y_true, preds_eval)
# 7. eval_results = coco_metric.result();  mAP50 = float(eval_results['MaP@.50IOU'])
# Required: model_eval, coco_metric, eval_results, mAP50

model_eval   = None  # <-- replace
coco_metric  = None  # <-- replace
eval_results = None  # <-- replace
mAP50        = None  # <-- replace

print('eval_results keys:', None if eval_results is None else list(eval_results.keys()))
print('mAP@0.50         :', mAP50)

### 4.4 AP@0.50 for `person`  *(10 pts)*

Compute Average Precision for the **`person`** class (id = 14) following the
Pascal VOC 2010+ protocol:

1. Re-run with `EVAL_THR = 0.05` to capture the full P/R curve.
2. Collect all `person` predictions across 10 images; sort by confidence descending.
3. Greedy IoU matching (`IOU_THR = 0.50`): label each detection TP or FP.
   A GT box can only be matched once.
4. Compute cumulative precision and recall at each step.
5. VOC monotone interpolation: smooth precision right-to-left with a running max,
   then `AP = Σ (recall[i+1] − recall[i]) × precision[i+1]` for unique recall changes.
6. Plot the P/R curve.

In [ ]:
# Given — do not modify
def box_iou_matrix(a, b):
    """Pairwise IoU between arrays a (M,4) and b (N,4) in xyxy format."""
    x1    = np.maximum(a[:, 0:1], b[:, 0]);   y1 = np.maximum(a[:, 1:2], b[:, 1])
    x2    = np.minimum(a[:, 2:3], b[:, 2]);   y2 = np.minimum(a[:, 3:4], b[:, 3])
    inter = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)

In [ ]:
# TODO (4.4a): collect person detections and assign TP / FP
# Required: person_dets (list of dicts with 'img_i','conf','box','is_tp'),
#           n_gt_person (int)

IOU_THR   = 0.50
EVAL_THR  = 0.05
PERSON_ID = CMAP_INV['person']

person_dets = None  # <-- replace
n_gt_person = None  # <-- replace

if person_dets is not None:
    n_tp = sum(1 for d in person_dets if d.get('is_tp'))
    print(f'{len(person_dets)} person detections: {n_tp} TP, {len(person_dets)-n_tp} FP')
print(f'GT person boxes: {n_gt_person}')

In [ ]:
# TODO (4.4b): AP computation and P/R plot
# Required: ap_person (float)

ap_person = None  # <-- replace

print(f'AP@0.50 (person) = {ap_person}')

## Auto-grader

Run the cell below to check your score.

In [ ]:
from grader_objectDetect import grade
grade(globals())